[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/state_space/state_space.ipynb)

# Heat Capacity Under a Constraint

This example computes how much a gas warms when it takes in heat under a constraint. A gas in a cylinder whose piston is latched keeps its volume, and all the heat it takes in warms it; under a piston loaded with a weight it keeps its pressure instead, and part of the heat goes into lifting the weight, so the same heat warms it less. The heat it takes per degree, its heat capacity, depends on what is held fixed.

A state of the gas is modelled as an antivector of the plane spanned by its entropy, `mv.s.dual()`, and its volume, `mv.v.dual()`, and a quantity is read from a state by pairing, `mv.s & state`. The gas is modelled by its energy as a function of the state: the energy's derivative is a reading of changes of state, a map `Scalar <- State`, whose values on the entropy and the volume are the temperature and minus the pressure, and the second derivative says how those change. A constraint is one more reading of the change of state, one that must stay zero. Stacked with the temperature's change, the two readings are one map from a change of state to both their changes; solving it for one degree warmer and the constraint held gives the heating direction, whatever the constraint is.

In the partial-derivative notation of thermodynamics, where a subscript names the quantity held fixed, the heat taken in is $\delta Q = T\,dS$, and the heat capacities at fixed volume and at fixed pressure are $C_V = T(\partial S/\partial T)_V$ and $C_p = T(\partial S/\partial T)_p$; for an ideal gas $C_p - C_V = nR$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from functools import partial

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.state_space import render

ga = Algebra("s0v0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
State = ga.gatype.antivector()                             # Antivector-
Reading = ga.gatype((Scalar, State))                       # Scalar+ <- State-

# The two readings a state is described by; entropy is in nR, volume and temperature in their initial values, and
# energy in nRT₀.
entropy = mv.s & State                                     # [] Scalar+ <- State-
volume = mv.v & State                                      # [] Scalar+ <- State-
# The gas's heat capacity at fixed volume, in nR.
thermal_capacity = 1.5

## Temperature and pressure from the energy

The ideal gas's energy grows exponentially with its entropy and falls as a power of its volume. Its derivative at a state, bound to a step of entropy, is the temperature, and bound to a step of volume minus the pressure; its second derivative, a form on pairs of changes, bound to one step gives the readings of how temperature and pressure change. The charts below hold the gas's states read two ways: temperature against entropy, with lines of equal volume in green and of equal pressure in purple, and pressure against volume, with isotherms in red and adiabats, lines of equal entropy, in teal. Both cylinders start at the dot.

In [ ]:
def energy(state: State) -> Scalar:
    """The energy of an ideal gas with a constant heat capacity at fixed volume."""
    return thermal_capacity * ((entropy(state) - volume(state).log()) / thermal_capacity).exp()

@jax.jit
def response(state: State) -> tuple[Scalar, Scalar, Reading, Reading]:
    """Temperature and pressure, and the readings of how they change."""
    differential = derivative(energy)(state)                             # [...] Scalar+ <- State-
    hessian = derivative(derivative(energy))(state)                      # [...] Scalar+ <- (State-, State-)
    return differential(mv.s.dual()), -differential(mv.v.dual()), hessian(mv.s.dual()), -hessian(mv.v.dual())

temperature_of = lambda states: response(states)[0]
pressure_of = lambda states: response(states)[1]
# Both cylinders at the reference state: unit volume and temperature, entropy measured from there.
start = (mv.s.dual() * 0 + mv.v.dual())[None].broadcast_to((2,))     # [cylinders] State-
render.draw_charts(temperature_of, pressure_of, start)

## The heating direction

The latched cylinder holds its volume, the reading `volume`; the loaded cylinder holds its pressure, read on a change of state by the pressure's change. The heating direction is the change of state per degree warmer that leaves the held reading unchanged, whatever that reading is, and the heat it takes in, the temperature times its entropy's change, is the heat capacity under that constraint. The arrows show the two directions: straight up along the line of equal volume, and along the line of equal pressure, which climbs more slowly.

In [ ]:
@jax.jit
def heating_direction(state: State, constraint: Reading) -> tuple[State, Scalar]:
    """The change of state per degree warmer with the constraint's reading held, and the heat it takes in."""
    temperature, _, temperature_change, _ = response(state)
    # The two readings of one unknown change: the temperature's rises by one, the constraint's stays put.
    changes = stack((temperature_change.broadcast_to(constraint.shape), constraint),
                    axis=-1).field(0)                                  # [...] (Scalar+ <- State-)[readings]
    requested = mv.scalar([[1], [0]]).field(0)                         # Scalar+[readings]
    direction = changes.solve(requested)                               # [...] State-
    return direction, temperature * entropy(direction)

def held(states: State) -> Reading:
    """What each cylinder holds `[cylinders]`: the latched one its volume, the loaded one its pressure."""
    _, _, _, pressure_change = response(states)
    return stack((volume, pressure_change[1]))

directions, heat_capacities = heating_direction(start, held(start))
render.draw_directions(temperature_of, pressure_of, start, directions)

In [ ]:
print("heat capacities, latched and loaded:", heat_capacities.to_array())

## Equal heat

Both cylinders take in the same heat in equal slices. Each slice moves a cylinder's state along its heating direction, taken at the middle of the slice, by as many degrees as the slice over the heat capacity. The heat each takes in is the area under its path on the temperature–entropy chart, down to zero temperature, and the shaded areas grow equally; the loaded piston's work is the area under its path on the pressure–volume chart. The third chart plots each cylinder's temperature against the heat it has taken in: straight lines whose slopes are one over the heat capacities.

This cell takes a few seconds, most of it drawing the animation.

In [ ]:
total_heat, n_steps, frame_duration_ms = 2.4, 48, 80

@jax.jit
def heated(states: State, heat: float) -> State:
    """The states after taking in a little heat each, along their heating directions, by the midpoint rule."""
    direction, capacity = heating_direction(states, held(states))
    halfway = states + direction * (heat / capacity / 2)
    direction, capacity = heating_direction(halfway, held(halfway))
    return states + direction * (heat / capacity)

def equal_heating():
    """The two cylinders' states, their heating directions and the heat each has taken in, as the slices of heat come
    in."""
    states = start
    for step in range(n_steps + 1):
        directions, _ = heating_direction(states, held(states))
        yield states, directions, step * total_heat / n_steps
        states = heated(states, total_heat / n_steps)

show_animation(render.heating(temperature_of, pressure_of, equal_heating()), "thermodynamics_heating",
               duration_ms=frame_duration_ms)

The latched gas, blue, warms along its line of equal volume and does no work; the loaded gas, orange, warms more slowly along its line of equal pressure, and its volume grows as it lifts the weight. Their heat capacities are constant for this gas, so equal heat warms them by its ratio to each: $1 + 2.4/1.5 = 2.6$ and $1 + 2.4/2.5 = 1.96$.

In [ ]:
end = list(equal_heating())[-1][0]                                   # [cylinders] State-
final_temperatures = temperature_of(end)                              # [cylinders] Scalar+

In [ ]:
print("final temperatures, latched and loaded:", final_temperatures.to_array())

In [ ]:
# checks
np.testing.assert_allclose(heat_capacities.to_array(), [thermal_capacity, thermal_capacity + 1], atol=1e-10)
np.testing.assert_allclose(final_temperatures.to_array(), [2.6, 1.96], rtol=1e-3)
# The energy taken in is the heat less the loaded piston's work against its fixed pressure.
work = pressure_of(start) * (volume(end) - volume(start))
np.testing.assert_allclose((energy(end) - energy(start) + work).to_array(), total_heat, rtol=1e-3)
np.testing.assert_allclose(volume(end[0]).to_array(), 1, atol=1e-10)